# Bacterial Growth Analysis

## Data Understanding


### Objective
The objective of this project is to analyse bacterial growth dynamics under different strains and temperature conditions.


### Research Question
How do bacterial strain, time and temperature influence bacterial population growth ?

### Dataset Context

This project uses a synthetic bacterial growth dataset designed
for data analysis and statistical modeling practice.

The dataset simulates bacterial population measurements under
different strains, temperatures, and time points.

Because the data is synthetic, the results of this analysis should
be interpreted as an analytical exercise rather than as evidence
from real biological experiments.


### Dataset Variables

| Variable | Description | Role |
|---|---|---|
| `experiment_id` | Unique identifier for each experiment | Identifier |
| `strain` | Bacterial strain used in the experiment | Experimental condition |
| `temperature` | Experimental temperature | Experimental condition |
| `hour` | Time elapsed since the beginning of the experiment, in hours | Time |
| `population` | Observed bacterial population at each time point | Response variable |
| `growth_rate` | Growth rate associated with the experimental condition | Model parameter |
| `capacity` | Carrying capacity associated with the experimental condition | Model parameter |




In [31]:
import pandas as pd 
import numpy as np

In [32]:
df = pd.read_csv("data/raw/bacteria_growth.csv")

df.tail()

,experiment_id,strain,temperature,hour,population,growth_rate,capacity
5934,87,C,20,16,107.0,0.007,1889
5935,46,B,20,14,877.0,0.158,42976
5936,57,B,25,23,96645.0,0.402,109742
5937,116,C,37,4,301.0,0.250,70000
5938,54,B,25,47,109073.0,0.402,109742


## Initial Dataset Inspection

The initial inspection confirms that the dataset contains the expected
variables: `experiment_id`, `strain`, `temperature`, `hour`,
`population`, `growth_rate`, and `capacity`.

The `hour` variable represents the temporal dimension of the
observations.

At this stage, no data quality conclusions are drawn. Structural and
temporal characteristics will be investigated in the following
sections.


In [33]:
df_sorted = df.sort_values(["experiment_id", "hour"])


## Temporal Structure

First, we inspect the minimum and maximum hour and the number of
observations available for each experiment.


In [34]:
df_sorted.groupby("experiment_id")["hour"].agg(
    ["min", "max", "count"]
).head(10)


,min,max,count
experiment_id,,,
1,0,48,50
2,0,48,50
3,0,48,49
4,0,48,49
5,0,48,49
6,0,48,50
7,0,48,49
8,0,48,51
9,0,48,49


## Temporal Ordering

Next, we check whether the observations within each experiment are
ordered chronologically.


In [35]:
df_sorted.groupby("experiment_id")["hour"].apply(
    lambda x: x.is_monotonic_increasing
).value_counts()


hour
True    120
Name: count, dtype: int64

## Temporal Findings

After sorting the dataset by `experiment_id` and `hour`, the temporal
ordering is consistent across all 120 experiments.

All experiments span from hour 0 to hour 48, corresponding to 49
distinct hourly time points.

The number of observations varies between experiments because some
experiments contain repeated time points. These repeated observations
are investigated in the following sections.


In [36]:
experiment_structure = df_sorted.groupby("experiment_id").agg(
    strain_unique=("strain", "nunique"),
    temperature_unique=("temperature", "nunique"),
    growth_rate_unique=("growth_rate", "nunique"),
    capacity_unique=("capacity", "nunique"),
    observations=("hour", "count")
)
experiment_structure

,strain_unique,temperature_unique,growth_rate_unique,capacity_unique,observations
experiment_id,,,,,
1,1,1,1,1,50
2,1,1,1,1,50
3,1,1,1,1,49
4,1,1,1,1,49
5,1,1,1,1,49
...,...,...,...,...,...
116,1,1,1,1,52
117,1,1,1,1,49
118,1,1,1,1,49


## Experimental Structure Findings

Within each experiment, `strain`, `temperature`, `growth_rate`, and
`capacity` each have a single unique value. This indicates that these
variables represent fixed experimental conditions within an experiment.

The number of observations varies between experiments. This variation
was investigated further through the temporal structure analysis and
duplicate checks.


In [37]:
df_sorted.groupby("experiment_id")["hour"].apply(
    lambda x: x.duplicated().sum()
).value_counts().sort_index()

hour
0    76
1    33
2     8
3     2
4     1
Name: count, dtype: int64

In [38]:
duplicated_hours = df_sorted[
    df_sorted.duplicated(
        subset=["experiment_id", "hour"],
        keep=False
    )
]

duplicated_hours.head(20)


,experiment_id,strain,temperature,hour,population,growth_rate,capacity
38,1,A,20,38,61546.0,0.256,65845
5890,1,A,20,38,61546.0,0.256,65845
61,2,A,20,12,1868.0,0.256,65845
5905,2,A,20,12,1868.0,0.256,65845
250,6,A,20,5,397.0,0.256,65845
5926,6,A,20,5,397.0,0.256,65845
368,8,A,20,25,25506.0,0.256,65845
5922,8,A,20,25,25506.0,0.256,65845
383,8,A,20,40,62901.0,0.256,65845
5913,8,A,20,40,62901.0,0.256,65845


## Duplicate Records

The initial temporal analysis identified duplicated `hour` values
within several experiments.

Further inspection shows that these duplicated observations contain
identical values across all dataset variables, including
`experiment_id`, `strain`, `temperature`, `hour`, `population`,
`growth_rate`, and `capacity`.

This indicates that the duplicated observations are duplicate
records rather than multiple independent measurements at the same
time point.

The duplicated records will be addressed during the data quality
assessment.


In [39]:
duplicate_count = df.duplicated().sum()
duplicate_rows = df[df.duplicated(keep=False)]

print(f"Number of duplicated records: {duplicate_count}")
print(f"Rows involved in duplication: {len(duplicate_rows)}")

Number of duplicated records: 59
Rows involved in duplication: 118


## Duplicate Findings

Each experiment contains a single unique value for `strain`,
`temperature`, `growth_rate`, and `capacity`. This indicates that
these variables represent fixed experimental conditions within each
experiment.

In contrast, the number of observations varies across experiments.
Since all experiments span from hour 0 to hour 48, corresponding to
49 distinct time points, experiments with more than 49 observations
contain additional records that require further investigation.

The temporal and duplicate-record analyses indicate that these
additional observations are associated with repeated time points.
Their treatment will be addressed during the data quality stage.


## Data Understanding Summary

The initial data understanding identified the following characteristics:

- The dataset contains seven variables describing bacterial growth
  experiments.
- Each `experiment_id` represents a fixed experimental condition.
- `strain`, `temperature`, `growth_rate`, and `capacity` remain
  constant within each experiment.
- `hour` and `population` vary across observations.
- All experiments span from hour 0 to hour 48.
- Temporal ordering is consistent after sorting by `experiment_id`
  and `hour`.
- The number of observations varies between experiments due to
  duplicated time points.
- An exact duplicate check identified 59 duplicated records,
  involving 118 rows in total.

These findings provide the basis for the subsequent data quality
assessment.


## Next Steps

The next stage will focus on data quality, including missing values,
duplicate records, duplicated time points, value ranges, and
consistency checks across experimental conditions.
